# Gëstukaay : test de la voix en wolof sur GPU NVIDIA

Ce notebook fait tourner **nos deux serveurs de voix tels quels** (`transcription/serveur.py` et `synthese/serveur.py`), avec les versions épinglées des images Docker (torch 2.11 avec CUDA 12.6). Il mesure :
- que les deux modèles se chargent sur le GPU (CUDA) ;
- la VRAM utilisée par les deux ensemble ;
- le temps d'une réponse vocale en wolof (Oolel) et celui d'une transcription (M-Kiriku), sur l'aller-retour.

**Avant de lancer** (Kaggle) :
1. *Settings* → *Accelerator* : **GPU T4 x2** (ou P100) ; *Internet* : **on**.
2. *Add-ons* → *Secrets* : ajouter `HF_TOKEN`, un jeton de lecture Hugging Face, après avoir accepté les conditions de https://huggingface.co/AIHubSN/M-Kiriku-ASR.
3. *Run all*. Durée : environ 15 minutes (installation et téléchargement d'environ 15 Go de modèles).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Deux environnements Python 3.12, exactement comme les images Docker (Kaggle est en 3.13)
!pip install -q uv
import os
os.environ["UV_HTTP_TIMEOUT"] = "300"
!uv venv -q -p 3.12 /kaggle/working/venv_tts && uv pip install -q --python /kaggle/working/venv_tts/bin/python \
    --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu126 \
    "torch==2.11.0+cu126" "torchaudio==2.11.0+cu126" "transformers==4.46.3" "diffusers==0.29.0" \
    "conformer==0.3.2" "librosa==1.0.0" "s3tokenizer==0.3.0" "safetensors==0.8.0" "huggingface_hub==0.36.2" \
    "omegaconf==2.3.1" "einops==0.8.2" "scipy==1.18.1" "numpy==2.5.3" "fastapi==0.142.2" "uvicorn==0.54.0"
!uv venv -q -p 3.12 /kaggle/working/venv_asr && uv pip install -q --python /kaggle/working/venv_asr/bin/python \
    --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu126 \
    "torch==2.11.0+cu126" "transformers==4.57.6" "av==19.0.1" "numpy==2.5.3" \
    "fastapi==0.142.2" "uvicorn==0.54.0" "python-multipart==0.0.32"
!/kaggle/working/venv_tts/bin/python -c "import torch; print('tts', torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!/kaggle/working/venv_asr/bin/python -c "import torch; print('asr', torch.__version__, torch.version.cuda, torch.cuda.is_available())"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/kaggle/working/modeles"
os.environ["TRANSCRIPTION_CLE"] = os.environ["SYNTHESE_CLE"] = "essai-kaggle"
print("jeton chargé (non affiché)")

In [ ]:
%%writefile serveur_transcription.py
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "fastapi>=0.115",
#     "uvicorn>=0.30",
#     "python-multipart>=0.0.9",
#     "torch",
#     "transformers<5",
#     "av>=12",
#     "numpy",
# ]
# ///
"""Service de transcription des notes vocales : M-Kiriku (décisions 0026, 0027, issue #28).

Format des API compatibles OpenAI, pour que le moteur l'appelle de la même façon où qu'il tourne
(carte graphique louée, Mac de KBD, PC d'Aziz) :

    POST /v1/audio/transcriptions   champ `file` (OGG/Opus de WhatsApp, WebM/Opus du site, WAV, MP3),
                                    `response_format` json | verbose_json ; réponse {"text", ...}
    GET  /ping                      {"statut": "pret"} une fois le modèle chargé
    GET  /v1/models

Lancer (le modèle est lu dans le cache Hugging Face, ~6 Go la première fois) :

    TRANSCRIPTION_CLE=<jeton choisi> uv run transcription/serveur.py --port 8100
    # puis, côté API Gëstukaay (.env) : TRANSCRIPTION_URL=http://<machine>:8100/v1
    #                                   TRANSCRIPTION_CLE=<le même jeton>

Sans TRANSCRIPTION_CLE, le service n'écoute que sur la machine elle-même (127.0.0.1) et refuse de
s'ouvrir au réseau : sur le wifi d'un hôtel ou d'une salle, n'importe qui pourrait s'en servir.
TRANSCRIPTION_MODELE change de modèle (défaut AIHubSN/M-Kiriku-ASR ; secours AIHubSN/Kiriku-Wolof-ASR).
TRANSCRIPTION_EVEIL_S (défaut 240, 0 = jamais) : après ce délai sans note, une seconde de silence est
transcrite pour garder le modèle chaud. Mesuré le 07/10 sur le Mac (MPS) : après une nuit sans note, la
première a pris 21 s (le moteur abandonne à 5 s), la suivante 1,7 s.
Rien n'est gardé : l'audio est décodé en mémoire puis oublié ; seuls la durée et le temps de calcul
sont journalisés.
"""

from __future__ import annotations

import argparse
import io
import logging
import os
import secrets
import threading
import time

import av
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Form, Header, HTTPException, UploadFile
from transformers import pipeline

TAUX = 16_000
DUREE_MAX_S = 61  # EF-11 : 60 s au plus
OCTETS_MAX = 5 * 1024 * 1024
MODELE = os.environ.get("TRANSCRIPTION_MODELE", "AIHubSN/M-Kiriku-ASR")
# Décodage glouton et court (mesuré : 16 s -> 1 s pour une question) ; horodatage au-delà de 30 s
GENERATION = {"max_new_tokens": 128, "num_beams": 1}

EVEIL_S = int(os.environ.get("TRANSCRIPTION_EVEIL_S", "240"))

journal = logging.getLogger("transcription")
app = FastAPI(title="Transcription Gëstukaay", version="1.0")
_asr = None
_verrou = threading.Lock()  # une seule inférence à la fois sur la carte : réveil et vraie note ne se croisent pas
_derniere = time.monotonic()  # dernière inférence (note ou réveil)


def _silence() -> dict:
    """Neuf à chaque appel : le pipeline vide le dictionnaire qu'on lui passe (« raw » retiré)."""
    return {"raw": np.zeros(TAUX, dtype=np.float32), "sampling_rate": TAUX}


def _inferer(entree: dict, **kw) -> dict:
    global _derniere
    with _verrou:
        try:
            return _asr(entree, generate_kwargs=GENERATION, **kw)
        finally:
            _derniere = time.monotonic()


def appareil() -> tuple[str, torch.dtype]:
    if torch.cuda.is_available():
        return "cuda", torch.float16
    if torch.backends.mps.is_available():
        return "mps", torch.float16
    return "cpu", torch.float32  # ~30 s par note : à éviter (0026)


def charger() -> None:
    global _asr
    nom, dtype = appareil()
    journal.warning("chargement de %s sur %s…", MODELE, nom)
    _asr = pipeline("automatic-speech-recognition", model=MODELE, device=nom, torch_dtype=dtype)
    _inferer(_silence())
    journal.warning("prêt.")


def _garder_chaud() -> None:
    """Fil de fond : une seconde de silence quand rien n'est passé depuis EVEIL_S (rien si occupé)."""
    while True:
        time.sleep(max(EVEIL_S / 4, 5))
        if time.monotonic() - _derniere >= EVEIL_S:
            t0 = time.perf_counter()
            _inferer(_silence())
            journal.warning("réveil du modèle en %.2f s", time.perf_counter() - t0)


def decoder(octets: bytes) -> np.ndarray:
    """N'importe quel conteneur audio -> float32 mono 16 kHz, en mémoire."""
    morceaux = []
    with av.open(io.BytesIO(octets)) as conteneur:
        flux = conteneur.streams.audio[0]
        reech = av.AudioResampler(format="flt", layout="mono", rate=TAUX)
        for paquet in conteneur.decode(flux):
            for trame in reech.resample(paquet):
                morceaux.append(trame.to_ndarray().reshape(-1))
        for trame in reech.resample(None):
            morceaux.append(trame.to_ndarray().reshape(-1))
    return np.concatenate(morceaux).astype(np.float32) if morceaux else np.zeros(0, dtype=np.float32)


def autoriser(authorization: str | None) -> None:
    attendu = os.environ.get("TRANSCRIPTION_CLE", "")
    if attendu and not (authorization and secrets.compare_digest(authorization, f"Bearer {attendu}")):
        raise HTTPException(401, "Clé absente ou invalide")


@app.get("/ping")
def ping() -> dict:
    if _asr is None:
        raise HTTPException(503, "chargement")
    return {"statut": "pret", "modele": MODELE}


@app.get("/v1/models")
def modeles() -> dict:
    return {"object": "list", "data": [{"id": "m-kiriku-asr", "object": "model", "owned_by": "AIHubSN"}]}


# Route synchrone : FastAPI la passe dans un fil à part, /ping répond pendant une transcription
@app.post("/v1/audio/transcriptions")
def transcrire(file: UploadFile, model: str = Form("m-kiriku-asr"),
               language: str | None = Form(None), response_format: str = Form("json"),
               authorization: str | None = Header(None)) -> dict:
    autoriser(authorization)
    if _asr is None:
        raise HTTPException(503, "Modèle en cours de chargement")
    octets = file.file.read(OCTETS_MAX + 1)
    if len(octets) > OCTETS_MAX:
        raise HTTPException(413, "Fichier trop gros")
    try:
        audio = decoder(octets)
    except Exception as e:  # noqa: BLE001 — audio illisible : 400, sans le contenu
        raise HTTPException(400, f"Audio illisible ({type(e).__name__})") from None
    finally:
        del octets
    duree = len(audio) / TAUX
    if duree > DUREE_MAX_S:
        raise HTTPException(400, "Note de plus de 60 s")
    t0 = time.perf_counter()
    texte = "" if duree < 0.3 else _inferer({"raw": audio, "sampling_rate": TAUX},
                                            return_timestamps=duree > 30)["text"].strip()
    journal.warning("note de %.1f s transcrite en %.2f s", duree, time.perf_counter() - t0)
    if response_format == "verbose_json":
        return {"text": texte, "duration": round(duree, 2), "language": language}
    return {"text": texte}


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Service de transcription M-Kiriku (0026).")
    p.add_argument("--hote", help="0.0.0.0 avec une clé, 127.0.0.1 sans clé (défaut)")
    p.add_argument("--port", type=int, default=8100)
    args = p.parse_args()
    logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(message)s")
    cle = os.environ.get("TRANSCRIPTION_CLE", "")
    args.hote = args.hote or ("0.0.0.0" if cle else "127.0.0.1")
    if not cle and args.hote not in ("127.0.0.1", "localhost", "::1"):
        p.error("sans TRANSCRIPTION_CLE, le service n'écoute que sur 127.0.0.1 (définir une clé pour l'ouvrir au réseau)")
    charger()
    if EVEIL_S > 0:
        threading.Thread(target=_garder_chaud, name="eveil", daemon=True).start()
    uvicorn.run(app, host=args.hote, port=args.port)


In [ ]:
%%writefile serveur_synthese.py
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "fastapi>=0.115",
#     "uvicorn>=0.30",
#     "torch",
#     "torchaudio",
#     "transformers==4.46.3",
#     "diffusers==0.29.0",
#     "conformer==0.3.2",
#     "librosa",
#     "s3tokenizer",
#     "safetensors",
#     "huggingface_hub",
#     "omegaconf",
#     "einops",
#     "scipy",
#     "numpy",
# ]
# ///
"""Service de voix de réponse : Oolel-Voices (Soynade Research), décision 0029, issue #29.

Oolel est utilisé TEL QUEL (licence AGPL-3.0 : https://huggingface.co/soynade-research/Oolel-Voices) ;
ce service ne fait que le charger et lui passer une phrase. Le découpage du texte, les silences, le
contrôle et l'encodage Opus sont faits par le moteur (`engine/…/synthese.py`).

Format des API compatibles OpenAI, pour que le moteur l'appelle de la même façon où qu'il tourne :

    POST /v1/audio/speech   JSON {"input": "<une phrase>", "response_format": "wav", "seed": 1}
                            -> WAV PCM 16 bits mono ; « seed » change le tirage (2e essai)
    GET  /ping              {"statut": "pret"} une fois le modèle chargé
    GET  /v1/models

Lancer (modèle lu dans le cache Hugging Face, ~8,5 Go la première fois) :

    SYNTHESE_CLE=<jeton choisi> uv run synthese/serveur.py --port 8200
    # puis, côté API Gëstukaay (.env) : SYNTHESE_URL=http://<machine>:8200/v1
    #                                   SYNTHESE_CLE=<le même jeton>

SYNTHESE_VOIX : enregistrement de la voix de référence (WAV, 10 à 20 s, accord écrit de la personne) ;
par défaut, la voix de la démo d'Oolel. Sans SYNTHESE_CLE, le service n'écoute que sur 127.0.0.1.
Rien n'est gardé : seuls la longueur du texte et le temps de calcul sont journalisés.
"""

from __future__ import annotations

import argparse
import io
import logging
import os
import secrets
import threading
import time
import wave

import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Response
from huggingface_hub import hf_hub_download, snapshot_download
from pydantic import BaseModel, Field
from transformers import AutoModel

MODELE = "soynade-research/Oolel-Voices"
REGLAGE = {"cfg_weight": 0.5, "exaggeration": 0.2, "temperature": 0.3}  # réglage prudent (0029)
CARACTERES_MAX = 600

journal = logging.getLogger("synthese")
app = FastAPI(title="Voix Gëstukaay", version="1.0")
_oolel = None
_voix = ""
_verrou = threading.Lock()  # une seule synthèse à la fois sur la carte graphique


class Demande(BaseModel):
    input: str = Field(min_length=1, max_length=CARACTERES_MAX)
    model: str = "oolel-voices"
    response_format: str = "wav"
    seed: int = 1


def appareil() -> str:
    if torch.cuda.is_available():
        return "cuda"
    return "mps" if torch.backends.mps.is_available() else "cpu"


def charger() -> None:
    global _oolel, _voix
    _voix = os.environ.get("SYNTHESE_VOIX") or hf_hub_download(
        "soynade-research/Oolel-Voices-Demo", "8_1_c.wav", repo_type="space")
    nom = appareil()
    journal.warning("chargement d'Oolel sur %s…", nom)
    _oolel = AutoModel.from_pretrained(snapshot_download(MODELE), trust_remote_code=True, device_map=nom)
    _oolel.generate("Salamaleekum", audio_prompt_path=_voix)  # chauffe
    journal.warning("prêt.")


def autoriser(authorization: str | None) -> None:
    attendu = os.environ.get("SYNTHESE_CLE", "")
    if attendu and not (authorization and secrets.compare_digest(authorization, f"Bearer {attendu}")):
        raise HTTPException(401, "Clé absente ou invalide")


@app.get("/ping")
def ping() -> dict:
    if _oolel is None:
        raise HTTPException(503, "chargement")
    return {"statut": "pret", "modele": MODELE}


@app.get("/v1/models")
def modeles() -> dict:
    return {"object": "list", "data": [{"id": "oolel-voices", "object": "model", "owned_by": "soynade-research"}]}


# Route synchrone : FastAPI la passe dans un fil à part, /ping répond pendant une synthèse
@app.post("/v1/audio/speech")
def parler(d: Demande, authorization: str | None = Header(None)) -> Response:
    autoriser(authorization)
    if _oolel is None:
        raise HTTPException(503, "Modèle en cours de chargement")
    if d.response_format != "wav":
        raise HTTPException(400, "response_format : wav seulement (l'Opus est fait par le moteur)")
    t0 = time.perf_counter()
    with _verrou:
        torch.manual_seed(d.seed)
        onde = _oolel.generate(d.input, audio_prompt_path=_voix, **REGLAGE).squeeze().float().cpu().numpy()
    pcm = (np.clip(onde, -1, 1) * 32767).astype("<i2").tobytes()
    journal.warning("%d caractères -> %.1f s en %.1f s", len(d.input), len(pcm) / 2 / _oolel.sr,
                    time.perf_counter() - t0)
    sortie = io.BytesIO()
    with wave.open(sortie, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(_oolel.sr)
        w.writeframes(pcm)
    return Response(sortie.getvalue(), media_type="audio/wav")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Service de voix Oolel (0029).")
    p.add_argument("--hote", help="0.0.0.0 avec une clé, 127.0.0.1 sans clé (défaut)")
    p.add_argument("--port", type=int, default=8200)
    args = p.parse_args()
    logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(message)s")
    cle = os.environ.get("SYNTHESE_CLE", "")
    args.hote = args.hote or ("0.0.0.0" if cle else "127.0.0.1")
    if not cle and args.hote not in ("127.0.0.1", "localhost", "::1"):
        p.error("sans SYNTHESE_CLE, le service n'écoute que sur 127.0.0.1 (définir une clé pour l'ouvrir au réseau)")
    charger()
    uvicorn.run(app, host=args.hote, port=args.port)


In [ ]:
import subprocess, time, requests
def journal(nom, n=4000):
    return open(f"{nom}.log").read()[-n:]
fichiers = {n: open(f"{n}.log", "w") for n in ("transcription", "synthese")}
procs = {
    "transcription": subprocess.Popen(["/kaggle/working/venv_asr/bin/python", "serveur_transcription.py", "--hote", "0.0.0.0", "--port", "8100"],
                                      stdout=fichiers["transcription"], stderr=subprocess.STDOUT, env=os.environ),
    "synthese": subprocess.Popen(["/kaggle/working/venv_tts/bin/python", "serveur_synthese.py", "--hote", "0.0.0.0", "--port", "8200"],
                                 stdout=fichiers["synthese"], stderr=subprocess.STDOUT, env=os.environ),
}
t0 = time.time()
prets = set()
while len(prets) < 2:
    for nom, port in (("transcription", 8100), ("synthese", 8200)):
        if nom in prets:
            continue
        if procs[nom].poll() is not None:  # arrêté : on montre pourquoi, et « Run all » s'arrête ici
            raise RuntimeError(f"{nom} s'est arrêté (code {procs[nom].returncode}). Journal :\n{journal(nom)}")
        try:
            if requests.get(f"http://localhost:{port}/ping", timeout=2).status_code == 200:
                prets.add(nom)
                print(f"{nom} prêt après {time.time() - t0:.0f} s (téléchargement compris)")
        except requests.ConnectionError:
            pass
    if time.time() - t0 > 1800:
        raise RuntimeError("plus de 30 min sans être prêts.\n" + "\n".join(f"== {n}\n{journal(n)}" for n in procs))
    time.sleep(5)
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
# Aller-retour : une réponse en wolof dite par Oolel, puis retranscrite par M-Kiriku
for nom, p in procs.items():
    if p.poll() is not None:
        raise RuntimeError(f"{nom} s'est arrêté. Journal :\n{journal(nom)}")
H = {"Authorization": "Bearer essai-kaggle"}
phrases = [
    "Am na ñaari milioŋ, ñeenti teemeeri junni ak juróom-benn fukki junni nit ci diiwaanu Cees ci atum ñaari junni ak ñaar-fukk ak ñett.",
    "Njëgu ceeb bi ci Cees, mi ngi tollu ci ñetti teemeer ak juróom-ñeent fukk ak benn ci kilo bi.",
]
for i, phrase in enumerate(phrases, 1):
    t0 = time.time()
    wav = requests.post("http://localhost:8200/v1/audio/speech", headers=H, json={"input": phrase, "response_format": "wav"}, timeout=300).content
    t_synth = time.time() - t0
    import io, wave
    with wave.open(io.BytesIO(wav)) as w:
        duree = w.getnframes() / w.getframerate()
    open(f"reponse_{i}.wav", "wb").write(wav)
    t0 = time.time()
    texte = requests.post("http://localhost:8100/v1/audio/transcriptions", headers=H,
                          files={"file": (f"reponse_{i}.wav", wav, "audio/wav")}, data={"response_format": "json"}, timeout=300).json()["text"]
    t_asr = time.time() - t0
    print(f"Phrase {i} : synthèse {t_synth:.1f} s pour {duree:.1f} s d'audio ; transcription {t_asr:.1f} s")
    print(f"   dit      : {phrase}\n   entendu  : {texte}")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
from IPython.display import Audio, display
display(Audio("reponse_1.wav"))

In [ ]:
# Journaux des deux serveurs (temps de calcul de chaque appel)
for n in ("transcription", "synthese"):
    print(f"===== {n}"); print(open(f"{n}.log").read()[-2500:])